# Marketing Campaign Response — Model Comparison and Business Decision

A concise synthesis of the completed classification experiments, with emphasis on imbalanced-data metrics, decision-threshold selection and campaign economics.

> **Portfolio note:** This notebook is a curated version of completed MSc coursework. The analytical code and saved outputs come from the original work; presentation, headings, file paths and explanatory text have been cleaned for portfolio use. The dataset itself is not redistributed here.


# Marketing Campaign Response Prediction

### Portfolio version

This notebook summarizes my classification work on customer response to a marketing campaign. The original coursework brief is not reproduced here.

**Objective:** estimate the probability of customer response and evaluate models using metrics that are appropriate for an imbalanced classification problem.


## 1. Why accuracy is not enough

Most customers do not respond to a campaign. In this setting, a classifier can achieve high overall accuracy while still missing many of the customers the campaign is actually trying to identify.

For that reason, the experiments emphasized:

- ROC-AUC
- precision
- recall
- F1 score
- log-loss
- decision-threshold tuning


## 2. Data preparation and feature engineering

The work included:

- imputation for missing numeric/categorical values;
- one-hot encoding of education and marital-status variables;
- scaling of continuous variables;
- log transforms and outlier clipping for monetary variables;
- RFM-style customer features;
- customer age;
- previous campaign-response indicators;
- experiments with PCA and reduced feature sets.

Preprocessing was implemented inside scikit-learn pipelines so transformations could be learned using training data only.


In [ ]:
# Representative pipeline pattern used across the experiments
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

numeric_features = ["Recency", "Frequency", "Monetary", "Age", "Income"]
categorical_features = ["Education", "Marital_Status"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(drop="first", handle_unknown="ignore"))
    ]), categorical_features)
])

example_model = Pipeline([
    ("preproc", preprocessor),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000))
])


## 3. Models explored

The completed experiments include:

- Logistic Regression
- Gaussian Naive Bayes
- Random Forest
- Extra Trees
- Multi-Layer Perceptron
- MLP with PCA

Hyperparameters were tuned using GridSearchCV and/or Optuna depending on the experiment.


In [ ]:
# Selected saved results from the completed notebooks
import pandas as pd

model_results = pd.DataFrame([
    {"Model": "Extra Trees", "Metric": "Holdout ROC-AUC", "Value": 0.904},
    {"Model": "Random Forest", "Metric": "Positive-class F1", "Value": 0.652},
    {"Model": "MLP + PCA", "Metric": "Threshold-tuned F1", "Value": 0.662},
    {"Model": "Normalized Logistic Regression", "Metric": "Positive-class Recall", "Value": 0.800},
])

model_results


## 4. Threshold optimization

A default threshold of 0.50 is not automatically the best business decision.

The MLP + PCA experiment achieved its strongest saved F1 score at a threshold of **0.26**:

| Metric | Result |
|---|---:|
| Precision | 61.04% |
| Recall | 72.31% |
| F1 | 0.662 |
| Simulated profit | €286 |

The profit value is specific to the contact-cost and successful-response value assumptions used in the experiment. The important methodological point is that the probability threshold was selected according to the decision objective rather than treated as fixed.


In [ ]:
# Generic threshold-evaluation pattern
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_threshold(y_true, probabilities, threshold):
    predictions = probabilities >= threshold
    return {
        "threshold": threshold,
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
    }

# Example:
# scores = [evaluate_threshold(y_test, y_prob, t)
#           for t in np.arange(0.05, 0.96, 0.01)]


## 5. Key modelling lesson

The experiments show why model choice and threshold choice should be separated.

For example:

- Extra Trees produced a strong saved ROC-AUC, indicating good ranking ability;
- Random Forest offered a strong balance of precision and recall at the default threshold;
- MLP + PCA became more useful after threshold tuning;
- normalized Logistic Regression provided high recall and a more interpretable linear benchmark.

This makes the project closer to a real campaign-selection problem than a simple accuracy competition.


## 6. What I would improve next

For a production-oriented extension I would:

- calibrate predicted probabilities;
- evaluate expected value on a fully untouched final test set;
- add lift/gain charts and top-decile response rates;
- compare threshold strategies under several campaign-cost scenarios;
- use SHAP or permutation importance for model interpretation;
- package preprocessing and scoring into a reproducible training script.
